# Avaliação do Agente — 50 perguntas por edital

Roda 50 perguntas em **memória off** (cada pergunta independente), captura tokens
de entrada/saída por turno e salva em XLSX. No fim, extrapola o custo para os
demais modelos pagos a partir dos tokens medidos.

**Antes de rodar:**
- Coloca este notebook em `evals/` (na raiz do projeto).
- Coloca os 3 xlsx de perguntas em `evals/perguntas/`.
- Garante que o `.env` da raiz tem `OPENAI_API_KEY`.


In [1]:
import os, sys, time
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_community.callbacks import get_openai_callback

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## Configuração

In [2]:
EDITAIS = {
    "bndes":     1,
    "cvm":       2,
    "petrobras": 3,
}

EDITAL_NOME = "petrobras"           # bndes | cvm | petrobras
PROVIDER    = "openai"
MODEL       = "gpt-4o-mini"

EDITAL_ID = EDITAIS[EDITAL_NOME]
ARQ_PERGUNTAS = Path("perguntas") / f"{EDITAL_NOME}.xlsx"

print(f"Edital:    {EDITAL_NOME} (id={EDITAL_ID})")
print(f"Modelo:    {PROVIDER}/{MODEL}")
print(f"Perguntas: {ARQ_PERGUNTAS}")

Edital:    petrobras (id=3)
Modelo:    openai/gpt-4o-mini
Perguntas: perguntas/petrobras.xlsx


## Carregar perguntas

In [3]:
df_q = pd.read_excel(ARQ_PERGUNTAS)
df_q = df_q.dropna(subset=["pergunta"])
df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)
print(f"{len(df_q)} perguntas carregadas")
df_q.head()

50 perguntas carregadas


,id,categoria,pergunta
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso PE...
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...


## Construir o agente

In [4]:
agente = build_agent(provider=PROVIDER, model=MODEL)
print(f"Agente: {PROVIDER}/{MODEL}")

Agente: openai/gpt-4o-mini


## Rodar as 50 perguntas

`chat_history=[]` em todas as chamadas → memória off. `get_openai_callback()`
captura tokens de **todas** as chamadas LLM dentro do escopo (invocação inicial
+ iterações de tool-call + eventual self-check).

In [5]:
resultados = []

for i, row in df_q.iterrows():
    pergunta = row["pergunta"]
    t0 = time.time()
    erro = None
    resposta = None

    try:
        with get_openai_callback() as cb:
            resposta = ask(
                agent=agente,
                question=pergunta,
                chat_history=[],          # memória OFF
                edital_id_ativo=EDITAL_ID,
            )
        in_tok    = cb.prompt_tokens
        out_tok   = cb.completion_tokens
        custo_usd = cb.total_cost
    except Exception as e:
        erro = str(e)
        in_tok = out_tok = 0
        custo_usd = 0.0

    latencia = round(time.time() - t0, 2)

    resultados.append({
        "id":            row["id"],
        "categoria":     row.get("categoria", ""),
        "pergunta":      pergunta,
        "resposta":      resposta,
        "input_tokens":  in_tok,
        "output_tokens": out_tok,
        "total_tokens":  in_tok + out_tok,
        "custo_usd":     round(custo_usd, 6),
        "latencia_s":    latencia,
        "erro":          erro,
    })

    flag = "ERRO" if erro else "ok"
    print(f"[{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
          f"in={in_tok:>5}  out={out_tok:>4}")

df_r = pd.DataFrame(resultados)
print(f"\n{len(df_r)} respostas coletadas")

Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32953.68it/s]


[ 1/50]   ok    9.0s  in= 8740  out= 105
[ 2/50]   ok   2.66s  in= 6041  out=  56
[ 3/50]   ok   3.48s  in= 6496  out= 139
[ 4/50]   ok    4.4s  in= 6235  out= 179
[ 5/50]   ok   3.35s  in= 7129  out= 110
[ 6/50]   ok   6.28s  in= 7081  out= 228
[ 7/50]   ok    3.3s  in= 7087  out= 112
[ 8/50]   ok   3.56s  in= 6024  out= 116
[ 9/50]   ok   4.33s  in= 6814  out= 178
[10/50]   ok   4.58s  in= 6460  out= 107
[11/50]   ok   6.24s  in=14190  out= 169
[12/50]   ok   6.55s  in=13511  out= 265
[13/50]   ok   3.78s  in=11388  out= 131
[14/50]   ok   4.92s  in=14476  out= 162
[15/50]   ok  10.75s  in=17783  out= 178
[16/50]   ok   8.65s  in=14779  out= 261
[17/50]   ok   4.46s  in= 6016  out= 215
[18/50]   ok  11.47s  in=16498  out= 425
[19/50]   ok   3.88s  in= 6845  out= 132
[20/50]   ok   4.58s  in= 6910  out= 139
[21/50]   ok   6.07s  in= 6106  out= 249
[22/50]   ok   6.76s  in= 6102  out= 268
[23/50]   ok   4.81s  in= 6296  out=  97
[24/50]   ok   9.52s  in= 6638  out= 141
[25/50]   ok    

## Salvar

In [6]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
slug_modelo = MODEL.replace("/", "_").replace(":", "_")
nome_base = f"{EDITAL_NOME}__{PROVIDER}__{slug_modelo}__{ts}"

Path("resultados").mkdir(exist_ok=True)
arq = Path("resultados") / f"{nome_base}.xlsx"
df_r.to_excel(arq, index=False)
print(f"Salvo: {arq}")

Salvo: resultados/petrobras__openai__gpt-4o-mini__20260429_060217.xlsx


## Sumário

In [7]:
total_in   = int(df_r["input_tokens"].sum())
total_out  = int(df_r["output_tokens"].sum())
total_cost = float(df_r["custo_usd"].sum())
n_erros    = int(df_r["erro"].notna().sum())
lat_media  = float(df_r["latencia_s"].mean())
lat_p95    = float(df_r["latencia_s"].quantile(0.95))

print(f"Perguntas:        {len(df_r)}  ({n_erros} com erro)")
print(f"Tokens entrada:   {total_in:,}")
print(f"Tokens saída:     {total_out:,}")
print(f"Tokens totais:    {total_in + total_out:,}")
print(f"Custo medido:     US$ {total_cost:.4f}   ({PROVIDER}/{MODEL})")
print(f"Latência média:   {lat_media:.1f}s")
print(f"Latência p95:     {lat_p95:.1f}s")

Perguntas:        50  (0 com erro)
Tokens entrada:   500,099
Tokens saída:     9,165
Tokens totais:    509,264
Custo medido:     US$ 0.0730   (openai/gpt-4o-mini)
Latência média:   7.7s
Latência p95:     19.1s


## Estimativa de custo para os outros modelos

Multiplica os tokens medidos pelo preço de cada modelo.

> **Caveats:**
> - Tokenizers diferem entre modelos. Claude 4.7, por exemplo, usa tokenizer novo
>   que conta ~35% mais tokens para o mesmo texto. Estimativa = ordem de grandeza.
> - Modelos com raciocínio explícito (Opus, GPT-5.5) podem produzir mais tokens de
>   saída que o gpt-4o-mini no mesmo turno; a estimativa de output é piso.
> - Preços marcados `APROX` precisam ser conferidos antes de decidir orçamento.
> - Coluna `3 editais` assume volume similar entre os três PDFs (BNDES é o menor;
>   CVM é o maior — então o número é uma média grosseira).

In [8]:
# preços em USD por 1M tokens — (input, output)
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.60),   # confirmado
    "openai/gpt-5.4-mini":            (0.25,  2.00),   # APROX -- verificar
    "openai/gpt-5.4":                 (1.25, 10.00),   # APROX -- verificar
    "openai/gpt-5.5":                 (5.00, 30.00),   # confirmado
    "google/gemini-2.5-flash-lite":   (0.10,  0.40),   # confirmado
    "google/gemini-3-flash-preview":  (0.30,  2.50),   # APROX -- verificar
    "google/gemini-3.1-pro-preview":  (1.50, 10.00),   # APROX (base <200k) -- verificar
    "anthropic/claude-haiku-4-5":     (1.00,  5.00),   # confirmado
    "anthropic/claude-sonnet-4-6":    (3.00, 15.00),   # confirmado
    "anthropic/claude-opus-4-7":      (5.00, 25.00),   # confirmado
}

linhas = []
for nome, (pi, po) in PRECOS.items():
    custo_1ed = total_in / 1_000_000 * pi + total_out / 1_000_000 * po
    linhas.append({
        "modelo":                nome,
        "preco_in_usd_p_1M":     pi,
        "preco_out_usd_p_1M":    po,
        "custo_50q_1edital":     round(custo_1ed, 4),
        "custo_50q_3editais":    round(custo_1ed * 3, 4),
    })

df_custos = (
    pd.DataFrame(linhas)
    .sort_values("custo_50q_1edital")
    .reset_index(drop=True)
)
df_custos

,modelo,preco_in_usd_p_1M,preco_out_usd_p_1M,custo_50q_1edital,custo_50q_3editais
0,google/gemini-2.5-flash-lite,0.10,0.4,0.0537,0.1610
1,openai/gpt-4o-mini,0.15,0.6,0.0805,0.2415
2,openai/gpt-5.4-mini,0.25,2.0,0.1434,0.4301
3,google/gemini-3-flash-preview,0.30,2.5,0.1729,0.5188
4,anthropic/claude-haiku-4-5,1.00,5.0,0.5459,1.6378
5,openai/gpt-5.4,1.25,10.0,0.7168,2.1503
6,google/gemini-3.1-pro-preview,1.50,10.0,0.8418,2.5254
7,anthropic/claude-sonnet-4-6,3.00,15.0,1.6378,4.9133
8,anthropic/claude-opus-4-7,5.00,25.0,2.7296,8.1889
9,openai/gpt-5.5,5.00,30.0,2.7754,8.3263


## Inspecionar respostas

In [9]:
for _, row in df_r.head(5).iterrows():
    print(f"--- Q{int(row['id'])} ---")
    print(f"P: {row['pergunta']}")
    print(f"R: {row['resposta']}")
    print(f"   tokens={row['total_tokens']}  latencia={row['latencia_s']}s\n")

--- Q1 ---
P: Qual o período de inscrição para o concurso PETROBRAS?
R: O período de inscrição para o concurso da Petrobras foi de 17 de dezembro de 2021 a 5 de janeiro de 2022, das 10 horas do primeiro dia às 18 horas do último dia (horário oficial de Brasília/DF).

ref: edital [5.2, Anexo IV]
   tokens=8845  latencia=9.0s

--- Q2 ---
P: Qual o valor da taxa de inscrição do concurso PETROBRAS?
R: O valor da taxa de inscrição para o concurso da Petrobras é de R$ 79,83.

ref: edital [5.1]
   tokens=6097  latencia=2.66s

--- Q3 ---
P: Em qual site o candidato deve fazer a inscrição do concurso PETROBRAS?
R: O candidato deve realizar a inscrição para o concurso da PETROBRAS exclusivamente pela internet, no seguinte endereço eletrônico: [http://www.cebraspe.org.br/concursos/petrobras_21_ns](http://www.cebraspe.org.br/concursos/petrobras_21_ns). 

Essa inscrição deve ser feita durante o período estabelecido no cronograma do edital. O comprovante de inscrição também estará disponível nesse m